# Model Selection, Training, and Evaluation

To predict API (Air Pollution Index) levels, we adopted a combination of regression, classification, and time-series forecasting models, selected based on the nature of the prediction task.

## 1. Regression Models

Goal: Predict the exact API value

- Linear Regression  
- Lasso Regression  
- Decision Tree Regressor  
- Random Forest Regressor  

Training Setup:

1. Models were trained twice:  
   - Without lag features: To estimate API based on current input only  
   - With lag features: To enable short-term forecasting  
2. Hyperparameter Tuning: Performed using `GridSearchCV` with 5-fold cross-validation (`cv=5`) and `scoring='neg_mean_squared_error'`  
3. Evaluation Metric: Root Mean Squared Error (RMSE) on the test set  

## 2. Classification Models

Goal: Predict the API level category (e.g., Good, Moderate, Unhealthy)

- Logistic Regression  
- Naive Bayes  
- Decision Tree Classifier  
- Random Forest Classifier  
- K-Nearest Neighbours (KNN)  

Training Setup:

1. Similarly trained with and without lag features  
2. Hyperparameter Tuning: `GridSearchCV` with 5-fold cross-validation and `scoring=mcc_scorer`  
   - *(where `mcc_scorer` refers to a custom or built-in scorer based on the Matthews Correlation Coefficient)*  
3. Evaluation Metric: Matthews Correlation Coefficient (MCC) on the test set  


## 3. Time-Series Forecasting

Goal: Predict future API values based on past trends

Model: Long Short-Term Memory (LSTM)

Training Setup:

- No cross-validation due to the sequential nature of time-series data  
- Trained using multiple windowing strategies:  
  - 1-hour window
  - 24-hour window

Forecasting Styles:
- Autoregressive: Model predicts next time step and feeds it back recursively  
- Recursive: Uses its own prior predictions as inputs for future steps


## Key Insights

1. Across both regression and classification tasks, the inclusion of lag features (particularly 1-hour API lag) dramatically improved model accuracy and predictive power.

**Regression (Best Model: XGBoost)**

|Inclusion of Lag Features  |No |Yes    |
|:-:|:-:|:-:|
|Top Feature|PM2.5 (importance ≈ 0.5)|API lag (1-hour) with importance ≈ 0.8|
|R²|0.356|> 0.98 |
|RMSE|10.2 |1.62 |

**Classification (Best Model: Random Forest)**

|Inclusion of Lag Features  |No |Yes    |
|:-:|:-:|:-:|
|Top Feature|PM2.5 (importance ≈ 0.12)|API lag (1-hour) with importance ≈ 0.5|
|MCC|0.1996 |0.96  |

Models trained with historical API values consistently outperformed those using only current pollutant and weather data. This highlights the importance of temporal dependencies in predicting both continuous API values and categorical air quality levels.

2. LSTM Models

LSTM models were constructed for individual stations, using both the input variables and the previous API value to predict the next API value. Two window intervals were used: 1 hour and 24 hours.

While the autoregressive predictions showed good performance, the recursive predictions generally behaved as follows:

- With a 1-hour window: Predictions were poor, resulting in significant errors and performance degradation.

- With a 24-hour window: Predictions were more accurate, with smaller errors, but tended to be unstable. This instability likely stems from the model's reliance on its own previous predictions over a longer horizon, causing small errors to accumulate and amplify over time, sometimes resulting in runaway values or invalid outputs (e.g., NaNs or Infs).

For LSTM-based time-series forecasting of API levels, autoregressive forecasting is more reliable, while recursive forecasting requires careful error handling—especially when using longer input windows.